# 🖥️ ComfyUI en Google Colab — CPU, sin Google Drive

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/I-s-ax/Comfyui_en_colab/blob/main/comfyui_colab_cpu_sin_drive.ipynb)

Esta versión ejecuta ComfyUI usando **solo CPU** y guarda todo en el almacenamiento temporal de la sesión de Colab, en `/content/ComfyUI`. **No conecta ni utiliza Google Drive.**

Incluye instalación, descarga de checkpoints/LoRA desde enlaces, listado de modelos e inicio de ComfyUI con enlace web.

**Importante:** al reiniciar o desconectar el entorno, los archivos de `/content` pueden perderse. Los modelos grandes tendrán que descargarse de nuevo. La CPU es lenta para generación de imágenes y no es recomendable para vídeo. Ejecuta las celdas en orden.

## 1. Instalar ComfyUI en el almacenamiento temporal de Colab

In [ ]:
#@title 🧰 Instalar ComfyUI para CPU (sin Drive)
import subprocess, sys
from pathlib import Path

WORKSPACE = "/content/ComfyUI"
repo = Path(WORKSPACE)

if (repo / ".git").exists():
    print("La instalación ya existe; se conservará para esta sesión.")
elif (repo / "main.py").exists():
    print("Ya existe una instalación de ComfyUI.")
else:
    if repo.exists() and any(repo.iterdir()):
        raise RuntimeError(f"{repo} existe y no parece una instalación de ComfyUI. Revisa esa carpeta.")
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", WORKSPACE], check=True)

if not (repo / "main.py").exists():
    raise FileNotFoundError("No encuentro main.py en /content/ComfyUI")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements.txt")], check=True)

for name in ["checkpoints", "vae", "loras", "controlnet", "upscale_models",
             "diffusion_models", "text_encoders", "clip_vision"]:
    (repo / "models" / name).mkdir(parents=True, exist_ok=True)

print("ComfyUI preparado en:", WORKSPACE)
print("Todo se guarda en el disco temporal de esta sesión, no en Google Drive.")

## 2. Descargar checkpoint, LoRA u otro modelo
Pega un enlace directo de descarga. En Civitai puedes introducir un token manual; no lo compartas ni publiques.

In [ ]:
#@title 📥 Descargar modelo
URL_MODELO = "" #@param {type:"string"}
CARPETA_DESTINO = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]
NOMBRE_ARCHIVO = "" #@param {type:"string"}
TOKEN_CIVITAI = "" #@param {type:"string"}

import os, time, requests
from pathlib import Path
from urllib.parse import urlparse, unquote

if not URL_MODELO.strip():
    raise ValueError("Pega primero el enlace de descarga.")
original_url = URL_MODELO.strip()
url = original_url
if "civitai.com" in urlparse(url).netloc.lower() and TOKEN_CIVITAI.strip():
    url += ("&" if "?" in url else "?") + "token=" + TOKEN_CIVITAI.strip()

dest = Path("/content/ComfyUI/models") / CARPETA_DESTINO
dest.mkdir(parents=True, exist_ok=True)
filename = NOMBRE_ARCHIVO.strip() or unquote(Path(urlparse(original_url).path).name)
if not filename or filename.lower() in ("download", "models"):
    filename = "modelo_descargado.safetensors"
target = dest / Path(filename).name
temp = target.with_name(target.name + ".part")

if target.exists() and target.stat().st_size:
    print(f"El archivo ya existe: {target} ({target.stat().st_size/1e9:.2f} GB)")
else:
    error = None
    for attempt in range(1, 4):
        try:
            with requests.get(url, stream=True, allow_redirects=True, timeout=(30, 120),
                              headers={"User-Agent":"ComfyUI-Colab-CPU/1.0"}) as response:
                if response.status_code != 200:
                    raise RuntimeError(f"HTTP {response.status_code}. Revisa el enlace/token.")
                if "text/html" in response.headers.get("Content-Type", "").lower():
                    raise RuntimeError("El servidor devolvió HTML, no el archivo del modelo.")
                total = int(response.headers.get("Content-Length", "0") or 0)
                got = 0
                with open(temp, "wb") as f:
                    for chunk in response.iter_content(8 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            got += len(chunk)
                            if total:
                                print(f"Descargado {got/1e9:.2f}/{total/1e9:.2f} GB", end="\r")
                if total and got != total:
                    raise RuntimeError(f"Descarga incompleta: {got}/{total} bytes")
                with open(temp, "rb") as f:
                    head = f.read(500).lower()
                if b"<html" in head or b"<!doctype" in head:
                    raise RuntimeError("El contenido parece HTML, no un modelo.")
                os.replace(temp, target)
                print(f"\nDescarga completa: {target} ({got/1e9:.2f} GB)")
                error = None
                break
        except Exception as e:
            error = e
            if temp.exists():
                temp.unlink()
            print(f"Intento {attempt}/3: {e}")
            if attempt < 3:
                time.sleep(attempt * 2)
    if error:
        raise RuntimeError(f"No se pudo descargar el modelo: {error}") from error
print("Carpeta:", dest)

## 3. Ver modelos disponibles

In [ ]:
#@title 🔎 Listar modelos
CARPETA_MODELOS = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "clip_vision"]
from pathlib import Path
folder = Path("/content/ComfyUI/models") / CARPETA_MODELOS
folder.mkdir(parents=True, exist_ok=True)
files = sorted(p for p in folder.iterdir() if p.is_file())
print("Carpeta:", folder)
if not files:
    print("No hay archivos aquí todavía.")
for p in files:
    print(f"{p.name} — {p.stat().st_size/1e9:.2f} GB")

## 4. Iniciar ComfyUI en CPU y obtener un enlace
La primera carga puede tardar varios minutos. Mantén esta celda ejecutándose y abre el enlace temporal en otra pestaña.

In [ ]:
#@title 🚀 Iniciar ComfyUI (CPU)
import subprocess, time, re, sys, urllib.request, shutil, threading
from pathlib import Path

repo = Path("/content/ComfyUI")
if not (repo / "main.py").exists():
    raise FileNotFoundError("Ejecuta primero la celda de instalación.")

def check_server():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188", timeout=3) as r:
            return r.status
    except Exception:
        return None

comfy = None
if check_server():
    print("ComfyUI ya está respondiendo en el puerto 8188.")
else:
    log_path = Path("/content/comfyui_cpu.log")
    log_file = open(log_path, "w", buffering=1)
    comfy = subprocess.Popen(
        [sys.executable, "-u", "main.py", "--cpu", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=str(repo), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    def stream_log():
        try:
            for line in comfy.stdout:
                log_file.write(line)
                print(line, end="", flush=True)
        finally:
            log_file.flush()
    threading.Thread(target=stream_log, daemon=True).start()
    print("Esperando a que ComfyUI inicie. No detengas esta celda...", flush=True)
    start = time.time()
    status = None
    while time.time() - start < 600:
        status = check_server()
        if status:
            print(f"\nComfyUI responde (HTTP {status}).", flush=True)
            break
        if comfy.poll() is not None:
            break
        time.sleep(3)
    if not status:
        try:
            print("\n===== FINAL DEL REGISTRO =====\n", Path("/content/comfyui_cpu.log").read_text(errors="ignore")[-6000:])
        finally:
            log_file.close()
        raise RuntimeError("ComfyUI no respondió en 10 minutos. Revisa el registro.")
    # Mantener el archivo de registro abierto para el hilo lector.

if not shutil.which("npm"):
    raise RuntimeError("No se encontró npm en este entorno de Colab.")
if not shutil.which("lt"):
    install = subprocess.run(["npm", "install", "-g", "localtunnel"],
                             text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(install.stdout[-2000:])
    if install.returncode != 0 or not shutil.which("lt"):
        raise RuntimeError("No se pudo instalar LocalTunnel.")

try:
    req = urllib.request.Request("https://loca.lt/mytunnelpassword", headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=15) as r:
        print("Contraseña de LocalTunnel (si la solicita):", r.read().decode().strip())
except Exception as e:
    print("No se pudo consultar la contraseña automáticamente:", e)

print("Creando enlace temporal; espera una URL https://....loca.lt", flush=True)
tunnel = subprocess.Popen(["lt", "--port", "8188"], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in tunnel.stdout:
    print(line, end="", flush=True)
    match = re.search(r"https://[A-Za-z0-9.-]+\.loca\.lt", line)
    if match:
        print("\nAbre este enlace en otra pestaña:", match.group(0), flush=True)
        print("No detengas esta celda mientras uses ComfyUI.", flush=True)

## Solución de problemas

- **Los modelos desaparecen:** es normal al reiniciar la sesión; el almacenamiento de Colab es temporal. Descárgalos de nuevo.
- **El modelo no aparece en ComfyUI:** colócalo en `/content/ComfyUI/models/checkpoints` o en la carpeta correspondiente y refresca la interfaz.
- **Generación lenta:** prueba 512 × 512, batch 1 y pocos pasos con un checkpoint ligero.
- **Vídeo:** no recomendado con CPU.
- **Enlace temporal:** puede fallar o caducar; vuelve a iniciar la celda para crear otro.
- **Civitai devuelve 401/403:** revisa que el enlace sea de descarga y que el token tenga acceso al modelo.